# DAIS 2026 — Deployment & Setup

Short checklist to get a workspace ready for the DAIS 2026 runbooks
(`Genie.ipynb`, `AgentBricks.ipynb`, `LakebaseApps.ipynb`, `MLflow.ipynb`).
All four run against a single `all`-target deployment.

### 1. Create Domains

In **Catalog → Discover → Domains → New domain**, create three domains and select the matching existing governed tag for each:

| Domain | Governed tag (select existing) |
|---|---|
| **Operations** | `caspers_domain_operations = true` |
| **Revenue & Customers** | `caspers_domain_revenue = true` |
| **Compliance & Safety** | `caspers_domain_compliance = true` |

### 2. Enable Data quality monitor.
Enable schema-level anomaly detection on `${CATALOG}.food_safety`

### 3. Add prompts schema to the experiments in MAS and others
Schema Catalog.prompts

### 4. Fix agent endpoint UC perms


In [ ]:
# EDIT THIS to match the catalog you passed to `bundle deploy --var catalog=...`
# (bundle default is `caspersdev`; this workspace uses something else if you
#  deployed with `--var catalog=<other>`).
CATALOG = ""  # e.g. "caspersdev"

assert CATALOG, "Set CATALOG above to the catalog name from your last bundle deploy."

result = dbutils.notebook.run(
    "../../utils/fix_agent_perms",
    1800,
    {
        "CATALOG": CATALOG,
        # Leave ENDPOINT_NAMES empty to use the default
        # f"{CATALOG}_{refund,complaint,support}_agent" pattern.
        "ENDPOINT_NAMES": "",
        "MAX_WAIT_SECONDS": "300",
    },
)
print(result)

### 5. Set up the Unity AI Gateway endpoint (`all` target only)

Both the Refund agent (LangGraph + `langchain_openai.ChatOpenAI`) and the
Complaint agent (DSPy + `dspy.LM('openai/...')`) on the `all` target route
their internal LLM calls through a single Unity AI Gateway (v2 Beta)
endpoint, so every agent LLM call gets PII guardrails, inference-table
audit, usage tracking, and rate limits applied centrally — and you can
attribute usage to either agent via the `requester` column on the
inference table.

The v2 Beta gateway is **UI-configured only** — there is no public REST /
SDK API for creating or modifying it (per
[Configure Unity AI Gateway endpoints](https://docs.databricks.com/aws/en/ai-gateway/configure-endpoints-beta)).
Do this manually in the workspace UI, **before** deploying the `all` target:

1. **Enable the preview**

   Account console → **Previews** → toggle **Unity AI Gateway** on. (Account
   admin only; skip if already enabled.)

2. **Create the endpoint**

   Workspace sidebar → **AI Gateway** → **Create Unity AI Gateway Endpoint**.

   - **Name**: `dais2026-ai-gateway` (or pick another name — pass it via
     `--params "AI_GATEWAY_ENDPOINT_NAME=<name>"` at deploy time).
   - **Primary model**: a foundation model whose tool-use is good. We use
     `databricks-claude-sonnet-4-5` to match the agents' default `LLM_MODEL`.
   - Click **Create**.

3. **Enable Inference Tables**

   Endpoint detail page → **Inference Tables** → **Edit** → enable.
   Point it at a UC schema you control (e.g. `<your-catalog>.ai_gateway`).
   This populates `<catalog>.ai_gateway.<endpoint>_payload` with one row
   per request — both allowed and blocked.

4. **Enable Usage Tracking**

   Endpoint detail page → **Usage Tracking** → **Edit** → enable. Per-request
   token counts land in `system.ai_gateway.usage` (account admins only —
   if your user can't query that table, ask an account admin to grant
   `SELECT ON SCHEMA system.ai_gateway` to your group).

5. **Configure Guardrails**

   Endpoint detail page → **Guardrails** → **Edit**. Enable:
   - **PII Detection** = **Block** (rejects requests containing SSNs, credit
     cards, etc — returns HTTP 400 before the LLM sees them).
   - **Jailbreak and Prompt Injection** = on.
   - **Unsafe Content** = on.

6. **Configure Rate Limits** (optional)

   Endpoint detail page → **Rate Limits** → **Edit**. Set a per-user QPM /
   TPM limit if you want to demo the burst-test in `MLflow.ipynb` →
   "Unity AI Gateway" section. Skip otherwise.

7. **Grant CAN_QUERY to `account users`**

   Endpoint detail page → **Permissions** → **Add user / group** →
   `account users` → **CAN_QUERY**.

   Why this is required and not auto-granted: v2 Beta gateway endpoints
   live on a separate API surface from regular serving endpoints, so the
   `mlflow.models.resources.DatabricksServingEndpoint(...)` resource
   listing that `agents.deploy()` uses to auto-grant CAN_QUERY does NOT
   work for them — it crashes with
   `NOT_FOUND: Dependent serving endpoint <gateway> does not exist`.
   Both the Refund and Complaint agent stages therefore **omit** the
   gateway from their `resources=[...]` lists and rely on you having
   granted CAN_QUERY manually here, so each deployed agent's runtime
   SP (a member of `account users`) can call the gateway at request
   time.

   If you'd rather scope the grant tighter than `account users`, after
   the first successful deploy grant CAN_QUERY explicitly to each
   service principal that owns the agent endpoints
   (`<catalog>_refund_agent`, `<catalog>_complaint_agent`) — Serving →
   endpoint → Permissions → check the creator.

8. **Deploy the `all` target with the gateway wired in**

   ```bash
   databricks bundle deploy -t all
   databricks bundle run caspers -t all \
     --params "CATALOG=<your-catalog>,AI_GATEWAY_ENDPOINT_NAME=dais2026-ai-gateway"
   ```

   When `AI_GATEWAY_ENDPOINT_NAME` is non-empty:
   - The Refund agent's `agent.py` builds a LangChain `ChatOpenAI`
     pointed at `<host>/ai-gateway/mlflow/v1`.
   - The Complaint agent's `agent.py` configures DSPy with
     `dspy.LM('openai/<gateway>', api_base=<host>/ai-gateway/mlflow/v1, ...)`.

   Both extract a fresh OAuth bearer from the runtime SP via
   `WorkspaceClient().config.authenticate()` at agent construction time
   (the SDK's `config.token` returns `None` in OAuth M2M mode, which is
   what `agents.deploy()` containers run under).

   When empty (the `default` and `complaints` targets, or the `all`
   target without that param), each agent falls back to its previous
   direct foundation-model endpoint plumbing — `ChatDatabricks` for
   Refund, `dspy.LM('databricks/<model>')` for Complaint.

> **Verify:** after deploy, send one request to each agent endpoint
> (`<catalog>_refund_agent`, `<catalog>_complaint_agent`) and check
> that two rows appear in `<catalog>.ai_gateway.<endpoint>_payload`.
> Both should show HTTP 200 with non-zero token counts.

### Pre-show warm-up

1. Open each runbook in the workspace and run its **pre-flight** cell
2. Click each app URL once (Ops Dashboard, Refund Manager) to dodge cold-start.
3. Click one sample question in each Genie space to warm `<catalog>-ops-warehouse` and `<catalog>-genie-warehouse`.
4. Open one Knowledge Assistant endpoint and the Supervisor endpoint to warm Model Serving.
5. Open all dashaboards
